In [1]:
import pandas as pd
import numpy as np
import xgboost as xg

In [2]:
%pip install xgboost
%pip install imbalanced-learn



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv('dataset.csv')
df.head()
df = df.drop(columns=['education'])

In [4]:
df.isnull().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [5]:
df.shape

(4240, 15)

In [6]:
df['TenYearCHD'].value_counts()

TenYearCHD
0    3596
1     644
Name: count, dtype: int64

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

# Hold out 20% as a final test set. Cross-validation runs on the other 80%.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(3392, 14) (848, 14)
TenYearCHD
0    2877
1     515
Name: count, dtype: int64


In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline            # imblearn's Pipeline, so the oversampler works
from imblearn.over_sampling import RandomOverSampler

def make_pipeline(model):
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('oversample', RandomOverSampler(random_state=42)),  # same as your resample(), training folds only
        ('model', model),
    ])

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

classifiers = {
    'RandomForest':       RandomForestClassifier(random_state=42),
    'AdaBoost':           AdaBoostClassifier(random_state=42),
    'GradientBoosting':   GradientBoostingClassifier(random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVC':                SVC(),
    'KNeighbors':         KNeighborsClassifier(),
    'DecisionTree':       DecisionTreeClassifier(random_state=42),
    'GaussianNB':         GaussianNB(),
    'XGBoost':            XGBClassifier(eval_metric='logloss', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

rows = []
for name, clf in classifiers.items():
    scores = cross_validate(make_pipeline(clf), X_train, y_train,
                            cv=cv, scoring=metrics, n_jobs=-1)
    row = {'Model': name}
    for m in metrics:
        row[m] = scores[f'test_{m}'].mean()
        row[f'{m}_std'] = scores[f'test_{m}'].std()
    rows.append(row)
    print(f"{name:20s} ROC-AUC {row['roc_auc']:.3f}  Recall {row['recall']:.3f}  F1 {row['f1']:.3f}")

RandomForest         ROC-AUC 0.704  Recall 0.138  F1 0.204
AdaBoost             ROC-AUC 0.709  Recall 0.658  F1 0.362
GradientBoosting     ROC-AUC 0.694  Recall 0.561  F1 0.360
LogisticRegression   ROC-AUC 0.731  Recall 0.682  F1 0.393
SVC                  ROC-AUC 0.705  Recall 0.577  F1 0.374
KNeighbors           ROC-AUC 0.636  Recall 0.507  F1 0.319
DecisionTree         ROC-AUC 0.552  Recall 0.243  F1 0.239
GaussianNB           ROC-AUC 0.720  Recall 0.299  F1 0.329
XGBoost              ROC-AUC 0.644  Recall 0.217  F1 0.245


In [10]:
results_df = pd.DataFrame(rows).sort_values('roc_auc', ascending=False).reset_index(drop=True)
results_df[['Model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(3)

,Model,accuracy,precision,recall,f1,roc_auc
0,LogisticRegression,0.679,0.276,0.682,0.393,0.731
1,GaussianNB,0.815,0.369,0.299,0.329,0.720
2,AdaBoost,0.648,0.250,0.658,0.362,0.709
3,SVC,0.707,0.277,0.577,0.374,0.705
4,RandomForest,0.839,0.420,0.138,0.204,0.704
5,GradientBoosting,0.697,0.265,0.561,0.360,0.694
6,XGBoost,0.797,0.283,0.217,0.245,0.644
7,KNeighbors,0.671,0.232,0.507,0.319,0.636
8,DecisionTree,0.767,0.236,0.243,0.239,0.552


In [11]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

best_name = results_df.loc[0, 'Model']   # highest CV ROC-AUC; pick by 'recall' or 'f1' if you prefer
print("Best model by CV:", best_name)

best_pipe = make_pipeline(classifiers[best_name])
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
if hasattr(best_pipe, 'predict_proba'):
    print("Test ROC-AUC:", roc_auc_score(y_test, best_pipe.predict_proba(X_test)[:, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best model by CV: LogisticRegression
Test accuracy: 0.6733490566037735
Test ROC-AUC: 0.6997013509288309

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.69      0.78       719
           1       0.25      0.60      0.36       129

    accuracy                           0.67       848
   macro avg       0.58      0.64      0.57       848
weighted avg       0.81      0.67      0.72       848

Confusion Matrix:
[[494 225]
 [ 52  77]]


In [12]:
for i in [10, 200, 110]:
    print("predicted class:", best_pipe.predict(X_test.iloc[[i]])[0],
          "| actual class:", y_test.iloc[i])

predicted class: 0 | actual class: 0
predicted class: 1 | actual class: 0
predicted class: 0 | actual class: 0


In [13]:
import pickle
# The pipeline includes the imputer and scaler, so there's no separate scaler.pkl
pickle.dump(best_pipe, open("chd_pipeline.pkl", 'wb'))

In [14]:
import pickle
with open("chd_pipeline.pkl", "rb") as file:
    model = pickle.load(file)

In [15]:
def predict(model, male, age, currentSmoker, cigsPerDay, BPMeds, prevalentStroke,
            prevalentHyp, diabetes, totChol, sysBP, diaBP, BMI, heartRate, glucose):
    yes = lambda v: 1 if str(v).lower() == "yes" else 0

    patient = pd.DataFrame([{
        'male': 1 if male.lower() == "male" else 0,
        'age': age,
        'currentSmoker': yes(currentSmoker),
        'cigsPerDay': cigsPerDay,
        'BPMeds': yes(BPMeds),
        'prevalentStroke': yes(prevalentStroke),
        'prevalentHyp': yes(prevalentHyp),
        'diabetes': yes(diabetes),
        'totChol': totChol,
        'sysBP': sysBP,
        'diaBP': diaBP,
        'BMI': BMI,
        'heartRate': heartRate,
        'glucose': glucose,
    }])[X.columns]   # same column order as training

    return model.predict(patient)[0]   # the pipeline imputes and scales automatically

In [16]:
# test 1
result = predict(model, male="female", age=56, currentSmoker="yes", cigsPerDay=3,
                 BPMeds="no", prevalentStroke="no", prevalentHyp="yes", diabetes="no",
                 totChol=285, sysBP=145, diaBP=100, BMI=30.14, heartRate=80, glucose=86)
print("The patient is at risk of heart disease" if result == 1
      else "The patient is not at risk of heart disease")

The patient is at risk of heart disease


In [17]:
# test 2
result = predict(model, male="female", age=63, currentSmoker="yes", cigsPerDay=3,
                 BPMeds="no", prevalentStroke="no", prevalentHyp="yes", diabetes="no",
                 totChol=267, sysBP=156.5, diaBP=92.5, BMI=27.1, heartRate=60, glucose=79)
print("The patient is at risk of heart disease" if result == 1
      else "The patient is not at risk of heart disease")

The patient is at risk of heart disease
